In [2]:
# =============================================================================
# Assignment: Topic Modeling with LDA
# Name: NIK AMIR FARIS BIN NIK AMIDI
# ID: SW01083709
# =============================================================================

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
import gensim
from gensim import corpora
from gensim.models.coherencemodel import CoherenceModel

# 1. Download necessary NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# 2. Read the data
# We load 'news_dataset.csv' and keep only the 'text' column
df = pd.read_csv('news_dataset.csv')
df = df[['text']]

# 3. Perform text pre-processing
# a. Remove null values
df = df.dropna(subset=['text'])

# Initialize tools
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Lowercasing
    text = str(text).lower()
    # Removing special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenization
    tokens = word_tokenize(text)
    # Removing stopwords and words shorter than 3 characters
    tokens = [t for t in tokens if t not in stop_words and len(t) > 3]
    # Stemming
    tokens = [stemmer.stem(t) for t in tokens]
    # Lemmatization
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

# Apply preprocessing to the 'text' column
processed_docs = df['text'].apply(preprocess_text)

# 4. Perform LDA using Gensim
# Create Dictionary
dictionary = corpora.Dictionary(processed_docs)

# Filter out words that appear in less than 5 documents or more than 50% of the docs
dictionary.filter_extremes(no_below=5, no_above=0.5)

# Create Corpus (Term Document Frequency)
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

# Train the LDA model with 4 topics
num_topics = 4
lda_model = gensim.models.LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=10,
    alpha='auto',
    per_word_topics=True
)

# Print the keywords for the 4 topics
print("Top keywords per topic:")
for idx, topic in lda_model.print_topics(-1):
    print(f'Topic: {idx} \nWords: {topic}\n')

# 5. Evaluate the LDA model using Coherence score
coherence_model_lda = CoherenceModel(
    model=lda_model, 
    texts=processed_docs, 
    dictionary=dictionary, 
    coherence='c_v'
)
coherence_lda = coherence_model_lda.get_coherence()
print(f'Coherence Score: {coherence_lda:.4f}')

# 6. Interpretation of the Coherence Score
"""
Interpretation:
The Coherence Score (using the c_v measure) evaluates how semantically similar 
the top words in each topic are. In this assignment, a score typically ranging 
between 0.4 and 0.7 indicates a reasonably interpretable model. A higher score 
suggests that the words within a topic belong to a consistent theme, making 
it easier for humans to label the topics (e.g., distinguishing between 'Autos' 
and 'Electronics'). If the score is low, it suggests that the topics may be 
overlapping or contain noisy, unrelated words, which might require further 
text cleaning or adjusting the number of topics.
"""

[nltk_data] Downloading package punkt to
[nltk_data]     /home/62af69a0-0330-4246-8917-
[nltk_data]     f16436100860/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/62af69a0-0330-4246-8917-
[nltk_data]     f16436100860/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/62af69a0-0330-4246-8917-
[nltk_data]     f16436100860/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/62af69a0-0330-4246-8917-
[nltk_data]     f16436100860/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Top keywords per topic:
Topic: 0 
Words: 0.012*"peopl" + 0.009*"would" + 0.006*"govern" + 0.005*"right" + 0.005*"think" + 0.005*"believ" + 0.004*"make" + 0.004*"state" + 0.004*"dont" + 0.004*"even"

Topic: 1 
Words: 0.010*"presid" + 0.007*"stephanopoulo" + 0.005*"american" + 0.005*"program" + 0.005*"year" + 0.005*"nation" + 0.005*"administr" + 0.004*"state" + 0.004*"univers" + 0.004*"govern"

Topic: 2 
Words: 0.012*"use" + 0.010*"maxaxaxaxaxaxaxaxaxaxaxaxaxaxax" + 0.010*"encrypt" + 0.010*"file" + 0.009*"system" + 0.008*"chip" + 0.006*"program" + 0.006*"window" + 0.005*"anonym" + 0.005*"inform"

Topic: 3 
Words: 0.011*"would" + 0.010*"dont" + 0.010*"like" + 0.009*"think" + 0.009*"know" + 0.007*"time" + 0.007*"well" + 0.006*"year" + 0.006*"good" + 0.006*"go"

Coherence Score: 0.5397


"\nInterpretation:\nThe Coherence Score (using the c_v measure) evaluates how semantically similar \nthe top words in each topic are. In this assignment, a score typically ranging \nbetween 0.4 and 0.7 indicates a reasonably interpretable model. A higher score \nsuggests that the words within a topic belong to a consistent theme, making \nit easier for humans to label the topics (e.g., distinguishing between 'Autos' \nand 'Electronics'). If the score is low, it suggests that the topics may be \noverlapping or contain noisy, unrelated words, which might require further \ntext cleaning or adjusting the number of topics.\n"